# 00 — Configuration (single source of truth)

Every other notebook in this project `%run`s this one, directly or through `10_document_framework`. **All project settings live here and nowhere else**: namespace, Volume layout, model services, the document metadata contract (including the configurable domain attributes), chunking, and retrieval settings.

This notebook is idempotent: running it creates only the dedicated schema and Unity Catalog Volume (with `landing/` and `staged_updates/`) and never touches unrelated workspace objects. Use a fresh `SCHEMA` — or `reset_project_data(confirm=True)` — for a fresh demo baseline.

| Setting | Meaning |
|---|---|
| `CATALOG` / `SCHEMA` / `VOLUME` | Project namespace; all tables, views, and source files live here |
| `LANDING_DIR` / `STAGED_DIR` | `landing/` is scanned by every ingestion pass; `staged_updates/` is a demo-only holding area never scanned by ingestion |
| `AI_MODE` | `auto` (use the configured native models) or `off` (no model calls) |
| `EMBEDDING_MODEL` / `CHAT_MODEL` | Databricks native model-service names; no external keys, installs, or provisioning |
| `CORE_METADATA_FIELDS` | Required identity fields for every domain: `document_id`, `version_id`, `document_date` |
| `DOMAIN_ATTRIBUTES` | Configurable domain metadata fields; agriculture default: `title`, `crop`, `region`, `topic` |
| `CHUNK_MAX_WORDS` / `CHUNK_OVERLAP_WORDS` | Section-aware chunking window settings |
| `DEFAULT_TOP_K` / `CANDIDATE_POOL_LIMIT` | Evidence chunks supplied per answer / candidates ranked before registry validation |

In [0]:
# --- Centralized configuration ----------------------------------------------------
# The single place to retarget the project. Running this notebook creates only a
# dedicated schema and a Unity Catalog Volume (plus its landing/ and
# staged_updates/ directories); unrelated workspace objects are never touched.
# Use a fresh SCHEMA for a fresh demo baseline, or reset_project_data(confirm=True).

# Namespace and Volume layout -----------------------------------------------------
CATALOG = ""  # Empty means use your currently selected catalog.
SCHEMA = "agri_retrieval_poc"
VOLUME = "documents"              # Unity Catalog Volume for source files
LANDING_DIR = "landing"           # scanned by every ingestion pass
STAGED_DIR = "staged_updates"     # demo-only holding area, never scanned by ingestion

# Model services (Databricks native; no external keys, installs, provisioning) ----
AI_MODE = "auto"  # auto: use the configured native models; off: no model calls
EMBEDDING_MODEL = "system.ai.gte-large-en"
CHAT_MODEL = "system.ai.gpt-oss-20b"
# Only the configured models are tried; there is no automatic paid-service provisioning.

# Document metadata contract ------------------------------------------------------
# CORE_METADATA_FIELDS are required for every domain. DOMAIN_ATTRIBUTES are
# configurable per deployment (agriculture is the default example): each must be
# a nonempty string in every source file, is stored with versions and chunks, and
# is retained as retrievable context. Order is preserved for validation messages
# and generated table schemas.
CORE_METADATA_FIELDS = ("document_id", "version_id", "document_date")
DOMAIN_ATTRIBUTES = ("title", "crop", "region", "topic")
METADATA_FIELDS = CORE_METADATA_FIELDS + DOMAIN_ATTRIBUTES

# Chunking and retrieval -----------------------------------------------------------
CHUNK_MAX_WORDS = 180      # body words per chunk window
CHUNK_OVERLAP_WORDS = 30    # overlap between consecutive windows
DEFAULT_TOP_K = 3          # evidence chunks supplied per answer
CANDIDATE_POOL_LIMIT = 20  # candidates ranked before registry validation

# --- Derived namespace, paths, and schemas (do not edit below) -------------------
assert AI_MODE in ("auto", "off"), "AI_MODE must be 'auto' or 'off'"
import re, json, uuid, threading
from pathlib import Path
from datetime import date, datetime, timezone, timedelta
from pyspark.sql import functions as F

for name in (EMBEDDING_MODEL, CHAT_MODEL):
    assert re.fullmatch(r"[A-Za-z0-9_.-]+", name), "Invalid model name"
if not CATALOG:
    CATALOG = spark.sql("SELECT current_catalog()").first()[0]
for name in (CATALOG, SCHEMA, VOLUME, LANDING_DIR, STAGED_DIR):
    assert name and "`" not in name, "Invalid identifier"
for field in METADATA_FIELDS:
    assert re.fullmatch(r"[a-z_][a-z0-9_]*", field), f"Invalid metadata field name: {field}"
NS = f"`{CATALOG}`.`{SCHEMA}`"  # backticked namespace used by every generated SQL statement
def table(name): return f"{NS}.`{name}`"

VERSION_SCHEMA = ", ".join(f"{f} string" for f in METADATA_FIELDS) + \
    ", body string, content_hash string, source_path string, filename string, ingested_at string"
CHUNK_SCHEMA = ", ".join(f"{f} string" for f in METADATA_FIELDS) + \
    ", content_hash string, source_path string, filename string, ingested_at string, chunk_id string, section_heading string, chunk_text string"
QUARANTINE_SCHEMA = "source_path string, content_hash string, reason string, observed_at string"
AUDIT_SCHEMA = "request_id string, question string, mode string, retrieved_json string, accepted_json string, rejected_count long, observed_at string"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NS}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {NS}.`{VOLUME}`")
ROOT = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
LANDING = ROOT / LANDING_DIR
STAGED = ROOT / STAGED_DIR
def ensure_volume_dirs():
    """Create landing/ and staged_updates/ (idempotent), retrying briefly.

    A freshly created Unity Catalog Volume can take a moment to appear on the
    /Volumes filesystem, so directory creation retries a few times instead of
    failing a run that executes immediately after reset_project_data().
    """
    import time
    for attempt in range(4):
        try:
            for path in (LANDING, STAGED): path.mkdir(parents=True, exist_ok=True)
            return
        except OSError:
            if attempt == 3: raise
            time.sleep(2)

ensure_volume_dirs()

# Serializes publication/retrieval within THIS notebook process only; it does not
# coordinate separate jobs, clusters, or other users (single-publisher assumption).
LOCK = threading.RLock()

def reset_project_data(confirm=False):
    """Drop and recreate THIS project's schema and Volume for a clean rerun.

    Scoped strictly to the configured <catalog>.<schema>: it refuses to run when
    SCHEMA is empty or 'default' and never touches anything else. Deletes the
    Volume (including landing/ and staged_updates/ fixtures) and every table,
    view, and stored embedding in the schema. Tables and views are recreated by
    running 10_document_framework immediately afterwards; then rerun the demo
    notebook to regenerate fixtures and data. Pass confirm=True to actually reset.
    """
    if not confirm:
        raise ValueError("reset_project_data is destructive; call with confirm=True")
    assert SCHEMA and SCHEMA != "default", "Refusing to reset the default schema"
    spark.sql(f"DROP SCHEMA IF EXISTS {NS} CASCADE")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NS}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {NS}.`{VOLUME}`")
    ensure_volume_dirs()
    print("Reset:", f"{CATALOG}.{SCHEMA}", "— now run 10_document_framework to recreate tables")

print("Configured:", f"{CATALOG}.{SCHEMA}", ROOT)